# Applying Scaling Factors to V03 Data

This notebook applies hourly V03-to-V04 scaling factors to the TEMPO V03 NO2 dataset. 

- observation times are converted from UTC to local time
- scaling factors are applied based on local hour
- observations before 10 AM local time are removed
- filtered to the Midwest study region and saved

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt

In [2]:
hour_scale_file ='/global/scratch/users/toireasamarie/Scaling Factors/low_CF_scaling_by_hour.parquet'
v03_file ='/global/scratch/users/toireasamarie/0.1x0.1_deg_gridded_datasets/gridded_and_crop_filtered/lowCF_TEMPO_03-07_0.1x0.1GCF_all_hours.parquet'

hour_scale =pd.read_parquet(hour_scale_file, engine='pyarrow')
v03_df =pd.read_parquet(v03_file, engine='pyarrow')

print('hour_scale shape:', hour_scale.shape)
print('v03_df shape:', v03_df.shape)

hour_scale shape: (12, 3)
v03_df shape: (3397014, 14)


In [3]:
hour_scale 

,hour,n,slope_origin
0,7,19375,0.831940
1,8,89095,0.875446
2,9,79812,0.894077
3,10,70072,0.866824
4,11,70794,0.855114
5,12,77357,0.834543
6,13,71772,0.813060
7,14,56134,0.770254
8,15,52372,0.739253
9,16,36088,0.728544


In [4]:
v03_df

,datetime,grid_lat,grid_lon,no2_mean,no2_std,no2_count,no2_sem,eff_cloud_fraction_mean,eff_cloud_fraction_std,amf_cloud_fraction_mean,amf_cloud_fraction_std,snow_ice_fraction_mean,snow_ice_fraction_std,cv_no2
0,2024-03-01 14:21:31.146651,39.55,-94.85,2.398081e+15,1.554050e+15,15,4.012539e+14,0.077335,0.013126,0.225971,0.029901,0.0,0.0,0.648039
1,2024-03-01 14:21:31.146651,41.25,-94.25,1.458990e+15,7.049008e+14,15,1.820046e+14,0.057213,0.006197,0.172525,0.016454,0.0,0.0,0.483143
2,2024-03-01 14:21:31.146651,41.55,-93.45,2.961572e+15,2.299921e+15,14,6.146796e+14,0.075453,0.012609,0.214683,0.028334,0.0,0.0,0.776588
3,2024-03-01 14:21:31.146651,41.65,-93.45,2.663766e+15,1.418604e+15,13,3.934500e+14,0.071391,0.012202,0.205748,0.030036,0.0,0.0,0.532556
4,2024-03-01 14:21:31.146651,41.65,-93.35,3.190018e+15,1.837509e+15,21,4.009774e+14,0.044535,0.026033,0.130932,0.068417,0.0,0.0,0.576018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3397009,2024-07-31 23:33:15.152400,43.75,-97.95,1.226792e+15,7.884975e+14,13,2.186899e+14,0.033057,0.025854,0.109373,0.074656,0.0,0.0,0.642731
3397010,2024-07-31 23:33:15.152400,43.75,-97.85,1.067047e+15,7.917712e+14,24,1.616196e+14,0.020792,0.009943,0.078556,0.035800,0.0,0.0,0.742021
3397011,2024-07-31 23:33:15.152400,43.85,-97.85,1.196948e+15,3.982585e+14,17,9.659187e+13,0.059991,0.025293,0.197631,0.069684,0.0,0.0,0.332728
3397012,2024-07-31 23:33:15.152400,44.85,-98.55,9.127368e+14,6.625882e+14,18,1.561735e+14,0.065685,0.015345,0.219664,0.043226,0.0,0.0,0.725936


In [5]:
v03_df['datetime'] = pd.to_datetime(v03_df['datetime']).dt.floor('h')

In [6]:
v03_df.columns

Index(['datetime', 'grid_lat', 'grid_lon', 'no2_mean', 'no2_std', 'no2_count',
       'no2_sem', 'eff_cloud_fraction_mean', 'eff_cloud_fraction_std',
       'amf_cloud_fraction_mean', 'amf_cloud_fraction_std',
       'snow_ice_fraction_mean', 'snow_ice_fraction_std', 'cv_no2'],
      dtype='object')

# adding timezone using timezonefinder

In [8]:
from timezonefinder import TimezoneFinder
from zoneinfo import ZoneInfo

In [10]:
tf =TimezoneFinder()

#timezone at each unique grid cell 
unique_coords = v03_df[['grid_lat', 'grid_lon']].drop_duplicates().copy()
unique_coords['tz_name'] = unique_coords.apply(lambda row: tf.timezone_at(lat=row['grid_lat'], lng=row['grid_lon']), axis=1)

print(f"unique grid cells: {len(unique_coords)}")
print(unique_coords['tz_name'].value_counts())

#merge timezone name into v3 df
v03_df = v03_df.merge(unique_coords, on=['grid_lat', 'grid_lon'], how='left')
v03_df['datetime_utc'] =pd.to_datetime(v03_df['datetime'])

#group by timezone, then apply time conversions
local_times = pd.Series(index=v03_df.index, dtype='datetime64[ns]')
for tz_name,group_idx in v03_df.groupby('tz_name').groups.items():
    tz = ZoneInfo(tz_name)
    utc_times = v03_df.loc[group_idx,'datetime_utc'].dt.tz_localize('UTC')
    local_times.loc[group_idx] =utc_times.dt.tz_convert(tz).dt.tz_localize(None)

v03_df['datetime_local'] =local_times
v03_df['local_hour'] =v03_df['datetime_local'].dt.hour.astype('int64')

print(v03_df[['datetime_utc', 'grid_lat', 'grid_lon', 'tz_name', 'datetime_local','local_hour']]
      .head(10).to_string(index=False))

unique grid cells: 9378
tz_name
America/Chicago                   8080
America/New_York                   407
America/Indiana/Indianapolis       379
America/Toronto                    245
America/Detroit                    111
America/Denver                      46
America/North_Dakota/New_Salem      39
America/Indiana/Vincennes           26
America/North_Dakota/Center         20
America/North_Dakota/Beulah         16
America/Indiana/Winamac              7
America/Indiana/Knox                 2
Name: count, dtype: int64
       datetime_utc  grid_lat  grid_lon         tz_name      datetime_local  local_hour
2024-03-01 14:00:00     39.55    -94.85 America/Chicago 2024-03-01 08:00:00           8
2024-03-01 14:00:00     41.25    -94.25 America/Chicago 2024-03-01 08:00:00           8
2024-03-01 14:00:00     41.55    -93.45 America/Chicago 2024-03-01 08:00:00           8
2024-03-01 14:00:00     41.65    -93.45 America/Chicago 2024-03-01 08:00:00           8
2024-03-01 14:00:00     41.65    -

# adding state info

In [11]:
import geopandas as gpd 
from shapely.geometry import Point

In [12]:
#us state shapefile
states = gpd.read_file('/global/scratch/users/toireasamarie/US_State_Shapefile/tl_2024_us_state.shp')[['NAME', 'geometry']].rename(columns={'NAME': 'state_name'})
states =states.to_crs('EPSG:4326')

#get unique grid cells and convert to geodf 
unique_coords = v03_df[['grid_lat', 'grid_lon']].drop_duplicates().copy()
unique_coords['geometry'] = unique_coords.apply(lambda row: Point(row['grid_lon'],row['grid_lat']),axis=1)
coords_gdf = gpd.GeoDataFrame(unique_coords,crs='EPSG:4326')

#spatial join and merge back to full df
coords_with_state = gpd.sjoin(coords_gdf, states, how='left', predicate='within')[['grid_lat', 'grid_lon', 'state_name']]
v03_df = v03_df.merge(coords_with_state, on=['grid_lat', 'grid_lon'], how='left')

print(v03_df['state_name'].value_counts())
print(f"\ngrid cells with no state assigned: {v03_df['state_name'].isna().sum()}")

ERROR 1: PROJ: proj_create_from_database: Open of /global/home/users/toireasamarie/.conda/envs/regrid_env2/share/proj failed


state_name
Iowa            501529
Illinois        454115
North Dakota    401884
Nebraska        371441
Minnesota       322470
South Dakota    311831
Missouri        249202
Kansas          213138
Indiana         159245
Ohio            109095
Kentucky         66573
Wisconsin        61241
Michigan         35640
Oklahoma         21486
Tennessee        21351
Arkansas         14914
Texas             7031
Colorado          2460
Name: count, dtype: int64

grid cells with no state assigned: 72368


In [13]:
v03_df

,datetime,grid_lat,grid_lon,no2_mean,no2_std,no2_count,no2_sem,eff_cloud_fraction_mean,eff_cloud_fraction_std,amf_cloud_fraction_mean,amf_cloud_fraction_std,snow_ice_fraction_mean,snow_ice_fraction_std,cv_no2,tz_name,datetime_utc,datetime_local,local_hour,state_name
0,2024-03-01 14:00:00,39.55,-94.85,2.398081e+15,1.554050e+15,15,4.012539e+14,0.077335,0.013126,0.225971,0.029901,0.0,0.0,0.648039,America/Chicago,2024-03-01 14:00:00,2024-03-01 08:00:00,8,Missouri
1,2024-03-01 14:00:00,41.25,-94.25,1.458990e+15,7.049008e+14,15,1.820046e+14,0.057213,0.006197,0.172525,0.016454,0.0,0.0,0.483143,America/Chicago,2024-03-01 14:00:00,2024-03-01 08:00:00,8,Iowa
2,2024-03-01 14:00:00,41.55,-93.45,2.961572e+15,2.299921e+15,14,6.146796e+14,0.075453,0.012609,0.214683,0.028334,0.0,0.0,0.776588,America/Chicago,2024-03-01 14:00:00,2024-03-01 08:00:00,8,Iowa
3,2024-03-01 14:00:00,41.65,-93.45,2.663766e+15,1.418604e+15,13,3.934500e+14,0.071391,0.012202,0.205748,0.030036,0.0,0.0,0.532556,America/Chicago,2024-03-01 14:00:00,2024-03-01 08:00:00,8,Iowa
4,2024-03-01 14:00:00,41.65,-93.35,3.190018e+15,1.837509e+15,21,4.009774e+14,0.044535,0.026033,0.130932,0.068417,0.0,0.0,0.576018,America/Chicago,2024-03-01 14:00:00,2024-03-01 08:00:00,8,Iowa
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3397009,2024-07-31 23:00:00,43.75,-97.95,1.226792e+15,7.884975e+14,13,2.186899e+14,0.033057,0.025854,0.109373,0.074656,0.0,0.0,0.642731,America/Chicago,2024-07-31 23:00:00,2024-07-31 18:00:00,18,South Dakota
3397010,2024-07-31 23:00:00,43.75,-97.85,1.067047e+15,7.917712e+14,24,1.616196e+14,0.020792,0.009943,0.078556,0.035800,0.0,0.0,0.742021,America/Chicago,2024-07-31 23:00:00,2024-07-31 18:00:00,18,South Dakota
3397011,2024-07-31 23:00:00,43.85,-97.85,1.196948e+15,3.982585e+14,17,9.659187e+13,0.059991,0.025293,0.197631,0.069684,0.0,0.0,0.332728,America/Chicago,2024-07-31 23:00:00,2024-07-31 18:00:00,18,South Dakota
3397012,2024-07-31 23:00:00,44.85,-98.55,9.127368e+14,6.625882e+14,18,1.561735e+14,0.065685,0.015345,0.219664,0.043226,0.0,0.0,0.725936,America/Chicago,2024-07-31 23:00:00,2024-07-31 18:00:00,18,South Dakota


# Removing hours before 10am (7, 8, 9)

In [14]:
v03_df_after10 = v03_df[v03_df["local_hour"] >= 10].copy()

In [15]:
v03_df_after10["local_hour"].value_counts().sort_index()

local_hour
10    308534
11    366832
12    420835
13    398356
14    346765
15    292489
16    251755
17    214391
18    103440
19      2008
Name: count, dtype: int64

# Apply scaling

In [16]:
#get local hour
v03_df_after10['local_hour'] =pd.to_datetime(v03_df_after10['datetime_local']).dt.hour.astype('int64')

#maje sure int64
hour_scale['hour']=hour_scale['hour'].astype('int64')

In [17]:
v03_df_after10.columns

Index(['datetime', 'grid_lat', 'grid_lon', 'no2_mean', 'no2_std', 'no2_count',
       'no2_sem', 'eff_cloud_fraction_mean', 'eff_cloud_fraction_std',
       'amf_cloud_fraction_mean', 'amf_cloud_fraction_std',
       'snow_ice_fraction_mean', 'snow_ice_fraction_std', 'cv_no2', 'tz_name',
       'datetime_utc', 'datetime_local', 'local_hour', 'state_name'],
      dtype='object')

In [18]:
#apply hour scaling 
hour_scale_rename = hour_scale[['hour', 'slope_origin']].rename(columns={'hour': 'local_hour', 'slope_origin': 'hour_scale_factor'})

v03_df_after10 = v03_df_after10.merge(hour_scale_rename, on='local_hour',how='left')
v03_df_after10['no2_scaled_by_hour'] =v03_df_after10['no2_mean'] * v03_df_after10['hour_scale_factor']

print('hour scaling done')
print('rows with no2_mean:', v03_df_after10['no2_mean'].notna().sum())
print('rows with no2_scaled_by_hour:', v03_df_after10['no2_scaled_by_hour'].notna().sum())

hour scaling done
rows with no2_mean: 2705405
rows with no2_scaled_by_hour: 2703397


In [19]:
2705405-2703397

2008

In [20]:
#flag rows that are within TEMPO hours but still missing a factor
tempo_hours = list(range(7, 19))
missing_states = (
    v03_df_after10[v03_df_after10['no2_mean'].notna() & v03_df_after10['local_hour'].isin(tempo_hours) &
    v03_df_after10['hour_scale_factor'].isna() & v03_df_after10['state_name'].notna()]['state_name'].unique())

if len(missing_states) > 0:
    print(f'missing factors for in-hours states: {missing_states}')
else:
    print('all rows matched ')

all rows matched 


In [21]:
#sample output 
print(v03_df_after10[['datetime_local', 'state_name', 'local_hour', 'no2_mean','hour_scale_factor', 'no2_scaled_by_hour']]
              # 'state_hour_scale_factor', 'no2_scaled_by_state_hour']]
      .dropna(subset=['no2_mean']).head(20).to_string(index=False))

     datetime_local state_name  local_hour     no2_mean  hour_scale_factor  no2_scaled_by_hour
2024-03-01 10:00:00   Michigan          10 5.637769e+15           0.866824        4.886952e+15
2024-03-01 10:00:00   Michigan          10 1.647404e+15           0.866824        1.428009e+15
2024-03-01 10:00:00      Texas          10 4.568050e+14           0.866824        3.959694e+14
2024-03-01 10:00:00      Texas          10 5.132411e+14           0.866824        4.448895e+14
2024-03-01 10:00:00      Texas          10 8.733302e+14           0.866824        7.570234e+14
2024-03-01 10:00:00      Texas          10 6.798386e+14           0.866824        5.893003e+14
2024-03-01 10:00:00      Texas          10 8.608021e+14           0.866824        7.461637e+14
2024-03-01 10:00:00      Texas          10 9.598780e+14           0.866824        8.320450e+14
2024-03-01 10:00:00      Texas          10 1.070887e+15           0.866824        9.282705e+14
2024-03-01 10:00:00      Texas          10 1.29615

In [22]:
v03_df_after10.columns

Index(['datetime', 'grid_lat', 'grid_lon', 'no2_mean', 'no2_std', 'no2_count',
       'no2_sem', 'eff_cloud_fraction_mean', 'eff_cloud_fraction_std',
       'amf_cloud_fraction_mean', 'amf_cloud_fraction_std',
       'snow_ice_fraction_mean', 'snow_ice_fraction_std', 'cv_no2', 'tz_name',
       'datetime_utc', 'datetime_local', 'local_hour', 'state_name',
       'hour_scale_factor', 'no2_scaled_by_hour'],
      dtype='object')

In [23]:
v03_df_after10['state_name'].unique()

array(['Michigan', 'Texas', 'Oklahoma', 'Kansas', 'Missouri', 'Nebraska',
       'Iowa', 'South Dakota', 'Wisconsin', nan, 'Minnesota',
       'North Dakota', 'Colorado', 'Illinois', 'Tennessee', 'Arkansas',
       'Kentucky', 'Indiana', 'Ohio'], dtype=object)

In [24]:
states_to_keep = ['Ohio', 'Kansas', 'Missouri', 'Indiana', 'Iowa', 'Illinois',
    'Wisconsin', 'Minnesota', 'Nebraska', 'South Dakota', 'North Dakota', 'Arkansas', 'Tennessee', 'Kentucky']

v03_to_save = v03_df_after10[v03_df_after10['state_name'].isin(states_to_keep)]

print(f"rows before filter: {len(v03_df_after10):,}")
print(f"rows after filter:  {len(v03_to_save):,}")
print(v03_to_save['state_name'].value_counts())

rows before filter: 2,705,405
rows after filter:  2,589,516
state_name
Iowa            398823
Illinois        340067
Nebraska        320234
North Dakota    314507
South Dakota    256359
Minnesota       244427
Missouri        193405
Kansas          184980
Indiana         129492
Ohio             86003
Kentucky         49225
Wisconsin        44881
Tennessee        15481
Arkansas         11632
Name: count, dtype: int64


In [25]:
out_file = '/global/scratch/users/toireasamarie/0.1x0.1_deg_gridded_datasets/final_datasets/a10_lowCF_hourscaled_tempo.parquet'
v03_to_save.to_parquet(out_file, engine='pyarrow', index=False)
print(f'saved: {out_file}')
print(f'Final shape: {v03_df.shape}')

saved: /global/scratch/users/toireasamarie/0.1x0.1_deg_gridded_datasets/final_datasets/a10_lowCF_hourscaled_tempo.parquet
Final shape: (3397014, 19)
